In [7]:
# Встановлення необхідних бібліотек (за потреби)
# !pip install langgraph langchain-openai langchain-core python-dotenv pydantic

import os
from datetime import datetime
from typing import TypedDict, Literal, Optional, List
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

# Завантаження змінних середовища
load_dotenv()

openrouter_base_url = os.getenv("OPENROUTER_BASE_URL")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if not openrouter_base_url or not openrouter_api_key:
    raise ValueError("OPENROUTER_BASE_URL or OPENROUTER_API_KEY is not set in .env")

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=openrouter_api_key,
    base_url=openrouter_base_url,
    temperature=0.1
)

print("Ініціалізація llm успішна.")

Ініціалізація llm успішна.


## 1. FAQ DB та типи даних для agent state

In [8]:
FAQ_DATABASE = {
    "password": [
        "Скидання пароля: Натисніть 'Забули пароль?' → введіть email → перевірте пошту",
        "Новий пароль: мінімум 8 символів з літерами та цифрами",
        "Лист не прийшов - перевірте папку спам"
    ],
    "payment": [
        "Перевірте дані картки та баланс",
        "Подвійне списання повернеться автоматично за 3-5 днів",
        "Зміна тарифу: Профіль → Підписка"
    ],
    "technical": [
        "Оновіть сторінку (F5) або очистьте кеш",
        "Спробуйте інший браузер",
        "Перевірте status.ourservice.com"
    ],
    "account": [
        "Перевірте правильність email",
        "Корпоративні акаунти: розділ 'Управління командою'",
        "Блокування - зверніться до підтримки"
    ],
    "general": [
        "Підтримка працює 24/7",
        "Час відповіді: 2-4 години"
    ]
}

class QueryClassification(TypedDict):
    """Структура класифікації як у конспекті"""
    type: Literal["problem", "question", "complaint"]
    category: Literal["password", "payment", "technical", "account", "general"]
    urgency: Literal["low", "medium", "high"]
    summary: str

class SupportAgentState(TypedDict):
    """Стан агента"""
    user_query: str
    classification: Optional[QueryClassification]
    search_results: Optional[List[str]]
    draft_response: Optional[str]
    needs_escalation: bool


## 2. Вузли LangGraph

4 послідовні вузли для проходу агентом:
1. `classify_query` — класифікація за типом, категорією, терміновістю та формування резюме.
2. `search_faq` — пошук 1-2 найрелевантніших пунктів із FAQ_DATABASE для визначеної категорії.
3. `draft_response` — формування короткої відповіді (до 50 слів) українською мовою з чіткими кроками.
4. `check_escalation` — перевірка критеріїв ескалації (терміновість 'high', тип 'complaint' або ключові слова) та фіналізація повідомлення.

In [9]:
class QueryClassificationModel(BaseModel):
    type: Literal["problem", "question", "complaint"] = Field(
        description="Тип запиту: "
                    "'problem' (проблема, несправність), "
                    "'question' (запитання, як зробити), "
                    "'complaint' (скарга, претензія, подвійне списання, тривалий збій)"
    )
    category: Literal["password", "payment", "technical", "account", "general"] = Field(
        description="Категорія запиту: "
                    "'password' (паролі), "
                    "'payment' (оплата, списання, підписка, зміна тарифу), "
                    "'technical' (робота сайту/сервісу), "
                    "'account' (користувачі, акаунти, корпоративний доступ), "
                    "'general' (загальні питання)"
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="Рівень терміновості: "
                    "'high' (вимоги повернення коштів, шахрайство, тривалий збій сайту), "
                    "'medium' (неможливість увійти/пароль), "
                    "'low' (стандартні запитання)"
    )
    summary: str = Field(description="Стисле резюме суті запиту українською мовою (1 речення)")

structured_classifier = llm.with_structured_output(QueryClassificationModel)


def classify_query(state: SupportAgentState) -> dict:
    """Вузол 1: Класифікація запиту користувача"""
    query = state["user_query"]

    prompt = f"""Класифікуй вхідний запит користувача служби технічної підтримки.
Запит: "{query}"

Правила:
- Зміна тарифного плану на дешевший/інший належить до категорії 'payment'.
- Проблеми з роботою/доступністю сайту належать до категорії 'technical'.
- Відновлення або забутий пароль належать до категорії 'password'.
- Додавання користувача чи управління командою належать до категорії 'account'.
- Емоційні скарги, звинувачення у шахрайстві, вимоги негайного повернення коштів або повідомлення про багатогодинні збої класифікуй як type='complaint' та urgency='high'.

Визнач тип, категорію, терміновість та надай стисле резюме."""

    res = structured_classifier.invoke(prompt)

    classification_data: QueryClassification = {
        "type": res.type,
        "category": res.category,
        "urgency": res.urgency,
        "summary": res.summary
    }

    return {"classification": classification_data}


class FAQSelectionModel(BaseModel):
    selected_faqs: List[str] = Field(
        description="1-2 найрелевантніші пункти FAQ зі списку для відповіді на запит користувача"
    )

structured_faq_selector = llm.with_structured_output(FAQSelectionModel)


def search_faq(state: SupportAgentState) -> dict:
    """Вузол 2: Пошук релевантних пунктів у базі знань FAQ для категорії"""
    query = state["user_query"]
    category = state.get("classification", {}).get("category", "general")

    available_faqs = FAQ_DATABASE.get(category, FAQ_DATABASE.get("general", []))

    prompt = (f"""Ти — модуль пошуку у базі знань FAQ. Обери від 1 до максимум 2 найрелевантніших пунктів із наданого списку для запиту користувача.

Запит: "{query}"
Категорія: {category}
Доступні пункти FAQ:
"""
              + "\n".join(f"- {item}" for item in available_faqs)
              + """

Обери 1 або 2 пункти, що найкраще відповідають запиту.""")

    try:
        res = structured_faq_selector.invoke(prompt)
        selected = res.selected_faqs[:2]
        if not selected:
            selected = available_faqs[:2]
    except Exception:
        selected = available_faqs[:2]

    return {"search_results": selected}


def draft_response(state: SupportAgentState) -> dict:
    """Вузол 3: Генерація стислої відповіді на основі знайденого FAQ"""
    query = state["user_query"]
    classification = state.get("classification", {})
    faq_results = state.get("search_results", [])

    faq_context = "\n".join(f"- {item}" for item in faq_results) if faq_results else "Немає пунктів FAQ"

    prompt = f"""Ти — агент технічної підтримки. Сформулюй стислу, чітку та інформативну відповідь на запит користувача, спираючись на знайдені пункти FAQ.

Вимоги:
1. Довжина: максимум 50 слів.
2. Тільки конкретні кроки та інструкції без привітань, вступів та зайвих слів.
3. Тільки українською мовою.
4. Якщо запит стосується подвійного списання коштів чи тривалого збою сайту, обов'язково поясни інформацію з FAQ (наприклад, термін автоматичного повернення 3-5 днів або перевірку status.ourservice.com) та додай, що запит передано спеціалісту для перевірки.

Запит користувача: {query}
Категорія: {classification.get('category')}
Знайдені пункти FAQ:
{faq_context}

Відповідь:"""

    res = llm.invoke(prompt)
    return {"draft_response": res.content.strip()}


def check_escalation(state: SupportAgentState) -> dict:
    """Вузол 4: Перевірка критеріїв ескалації та формування фінального тексту"""
    classification = state.get("classification") or {}
    query_lower = (state.get("user_query") or "").lower()

    urgency = classification.get("urgency", "low")
    q_type = classification.get("type", "question")

    escalation_keywords = ["шахрайство", "судов", "юридичн"]
    has_keywords = any(kw in query_lower for kw in escalation_keywords)

    needs_escalation = (urgency == "high") or (q_type == "complaint") or has_keywords

    current_response = state.get("draft_response", "")
    escalation_banner = "⚠️ Запит передано спеціалісту (відповідь за 24 год)."

    if needs_escalation:
        if escalation_banner not in current_response:
            final_response = f"{current_response}\n\n{escalation_banner}"
        else:
            final_response = current_response
    else:
        final_response = current_response

    return {
        "needs_escalation": needs_escalation,
        "draft_response": final_response
    }

## 3. Граф LangGraph (`build_support_agent`)

Створюємо строго послідовний граф:
   * START ->
   * classify_query ->
   * search_faq ->
   * draft_response ->
   * check_escalation ->
   * END

In [10]:
def build_support_agent():
    """Побудова та компіляція послідовного графа агента технічної підтримки"""
    workflow = StateGraph(SupportAgentState)

    # Додавання вузлів
    workflow.add_node("classify_query", classify_query)
    workflow.add_node("search_faq", search_faq)
    workflow.add_node("draft_response", draft_response)
    workflow.add_node("check_escalation", check_escalation)

    # Додавання послідовних ребер
    workflow.add_edge(START, "classify_query")
    workflow.add_edge("classify_query", "search_faq")
    workflow.add_edge("search_faq", "draft_response")
    workflow.add_edge("draft_response", "check_escalation")
    workflow.add_edge("check_escalation", END)

    return workflow.compile()

## 4. Тестування агента

Функція `calculate_answer_relevancy` оцінює якість відповіді за допомогою LLM (шкала 0-1).

Функція `test_agent` запускає агента для 5 тестових запитів і розраховує фінальні метрики:
- Task Completion Rate (TCR, ціль 100%)
- Average Answer Relevancy (ціль ≥ 80%)
- Escalation Rate (очікувано 40%, тобто 2/5 запитів)

In [11]:
# Оцінка релевантності
def calculate_answer_relevancy(query: str, response: str) -> float:
    """Оцінка релевантності"""

    relevancy_prompt = f"""
    Оцінка релевантності (0-1):
    Запит: {query}
    Відповідь: {response}

    Тільки число:
    """

    try:
        result = llm.invoke(relevancy_prompt)
        text = result.content.strip()
        score = float(''.join(c for c in text if c.isdigit() or c == '.'))
        return min(max(score, 0.0), 1.0)
    except:
        return 0.5

def test_agent():
    """Тестування агента на 5 запитах"""

    agent = build_support_agent()

    test_queries = [
        "Я забув пароль від свого акаунту, як його відновити?",
        "З мене двічі зняли гроші за підписку! Це шахрайство! Поверніть кошти негайно!",
        "Чи можу я змінити тарифний план на дешевший?",
        "Сайт не працює вже третю годину! Коли полагодите?",
        "Як додати нового користувача до корпоративного акаунту?"
    ]

    results = []
    relevancy_scores = []

    print("=" * 70)
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S} ТЕСТУВАННЯ АГЕНТА ТЕХНІЧНОЇ ПІДТРИМКИ")
    print("=" * 70)

    for i, query in enumerate(test_queries, 1):
        print(f"\n{'='*70}")
        print(f"📧 ТЕСТ {i}/5")
        print(f"{'='*70}")
        print(f"ЗАПИТ: {query}")
        print("-" * 70)

        initial_state = {
            "user_query": query,
            "needs_escalation": False
        }

        config = {"configurable": {"thread_id": f"test_{i}"}}

        try:
            result = agent.invoke(initial_state, config)
            results.append(result)

            # Виводимо класифікацію
            if result.get("classification"):
                cls = result["classification"]
                print(f"\n📊 КЛАСИФІКАЦІЯ:")
                print(f"   Тип: {cls.get('type')}")
                print(f"   Категорія: {cls.get('category')}")
                print(f"   Терміновість: {cls.get('urgency')}")
                print(f"   Резюме: {cls.get('summary')}")

            # Виводимо знайдені FAQ
            if result.get("search_results"):
                print(f"\n🔍 FAQ ({len(result['search_results'])} пунктів):")
                for j, faq in enumerate(result['search_results'], 1):
                    print(f"   {j}. {faq}")

            # ВИВОДИМО ВІДПОВІДЬ АГЕНТА
            if result.get("draft_response"):
                print(f"\n💬 ВІДПОВІДЬ АГЕНТА:")
                print("-" * 70)
                print(result["draft_response"])
                print("-" * 70)

                # Рахуємо релевантність
                relevancy = calculate_answer_relevancy(query, result["draft_response"])
                relevancy_scores.append(relevancy)
                print(f"\n📈 Релевантність: {relevancy:.0%}")

            # Статус ескалації
            if result.get("needs_escalation"):
                print("\n⚠️ СТАТУС: Ескаловано до спеціаліста")
            else:
                print("\n✅ СТАТУС: Оброблено автоматично")

        except Exception as e:
            print(f"❌ ПОМИЛКА: {e}")
            results.append({})

    # Фінальні метрики
    print(f"\n{'='*70}")
    print("ПІДСУМКОВІ МЕТРИКИ")
    print("=" * 70)

    # Task Completion Rate
    completed = sum(1 for r in results if r.get("draft_response"))
    tcr = completed / len(results) if results else 0
    print(f"✅ Task Completion Rate: {tcr:.0%} ({completed}/{len(results)})")

    # Answer Relevancy
    if relevancy_scores:
        avg_relevancy = sum(relevancy_scores) / len(relevancy_scores)
        print(f"🎯 Average Answer Relevancy: {avg_relevancy:.0%}")

    # Escalation Rate
    escalated = sum(1 for r in results if r.get("needs_escalation"))
    print(f"📊 Escalation Rate: {escalated}/{len(results)} ({escalated/len(results)*100:.0f}%)")

    print("=" * 70)

    return results

In [12]:
# Запуск тестування агента підтримки
results = test_agent()

2026-11-08 18:38:22 ТЕСТУВАННЯ АГЕНТА ТЕХНІЧНОЇ ПІДТРИМКИ

📧 ТЕСТ 1/5
ЗАПИТ: Я забув пароль від свого акаунту, як його відновити?
----------------------------------------------------------------------

📊 КЛАСИФІКАЦІЯ:
   Тип: question
   Категорія: password
   Терміновість: medium
   Резюме: Користувач запитує, як відновити забутий пароль від свого акаунту.

🔍 FAQ (2 пунктів):
   1. Скидання пароля: Натисніть 'Забули пароль?' → введіть email → перевірте пошту
   2. Лист не прийшов - перевірте папку спам

💬 ВІДПОВІДЬ АГЕНТА:
----------------------------------------------------------------------
Натисніть 'Забули пароль?' на сторінці входу, введіть свій email та перевірте пошту для отримання інструкцій. Якщо листа немає, перевірте папку спам.
----------------------------------------------------------------------

📈 Релевантність: 100%

✅ СТАТУС: Оброблено автоматично

📧 ТЕСТ 2/5
ЗАПИТ: З мене двічі зняли гроші за підписку! Це шахрайство! Поверніть кошти негайно!
-------------------------